In [1]:
# 02 notebook is for cleaning and filtering an eligible fasta

In [2]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

In [3]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# set up fasta
fasta_file = (
    project_root
    / "data"
    / "raw"
    / "atm_vertebrate_orthologs"
    / "ncbi_dataset"
    / "data"
    / "protein.faa"
)

records = list(SeqIO.parse(fasta_file, "fasta"))

print(f"first ID: {records[0].id}")
print(f"first description: {records[0].description}")
print(f"first 50 amino acids: {records[0].seq[:50]}")


first ID: XP_040832827.1
first description: XP_040832827.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
first 50 amino acids: MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRHSDSK


In [4]:
for record in records[:10]:
    print(record.description)


XP_040832827.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
XP_040832828.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
XP_040832829.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X2]
XP_040832830.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X3]
XP_059322021.1 ATM [organism=Ammospiza nelsoni] [GeneID=132069672]
XP_079784892.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=1]
XP_079784893.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=2]
XP_079784894.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=3]
XP_084165020.1 atm [organism=Notolepis coatsorum] [GeneID=311224077]
XP_044525056.1 ATM [organism=Gracilinanus agilis] [GeneID=123241475] [isoform=X1]


In [5]:
import re
import pandas as pd

def extract_field(field, description):
    pattern = rf"\[{field}=([^\]]+)\]"
    match = re.search(pattern, description)

    if match:
        return match.group(1)
    return pd.NA

# fasta list of dicts to csv
rows = []
for record in records:
    sequence = str(record.seq).upper()
    row = {
        "accession" : record.id,
        "gene_symbol" : record.description.split(maxsplit=2)[1],
        "organism"  : extract_field("organism", record.description),
        "gene_id"  : extract_field("GeneID", record.description),
        "isoform"  : extract_field("isoform", record.description),
        "sequence_length" : len(sequence),
        "ambiguous_x_count" :sequence.count("X"),
        "stop_count" :sequence.count("*"),
        "sequence" : sequence,
    }

    rows.append(row)

for _ in rows[:5]:
    print(_)

{'accession': 'XP_040832827.1', 'gene_symbol': 'ATM', 'organism': 'Ochotona curzoniae', 'gene_id': '121154090', 'isoform': 'X1', 'sequence_length': 3059, 'ambiguous_x_count': 0, 'stop_count': 0, 'sequence': 'MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRHSDSKQGKYLNWDAVFRFLQKYIQKETECQRTAKPNVSVSTQASRQKKMQELSSLVKYFIKCANKRAPRLKCQELLSYIMETVKDSSNSSTYGADCSNILLKDILSVRKYWCEISQQQWLELFSVYFTLYLKPSQDINRVLVARIIHAVTKGCCLQTDGLNSKYLDFFSKAVQYARQEKSSAGLSHILAALNIFLKTLAVNFRIRACELGDEILPVLLYIWAQHRLNDSLKEIIVELFQLQIYIHHPKGAKTQEKGAYISAKWQSILYNIYDLIVNEISHIGSRGKYTSGSRNIAVKENLIELMADICHQVFNEDTRSLEISQSYTITPRVSSDHSAPCKRRKIELGWEVIKDHLQKPQNDYDLVPWLQITTQLVSKYPASLPDHDLSSLLMILYQLLSQRRRGERTPYVLRCLTEVALCQGKKSNLETSQQSDLLKLWIKICTATFRGISSEQTQAENFGLLGAIIQGSLVEIDREFWKLFTGSACRPSCSSICCLALAMRSCIIPEAIHTGLELSICEVNGSFSLKELIMRWILCYRLEDDLEDRTELSPILHSNFPHFILEKILVSLTMKDCKAAMNFFQSGQECEHQKVKEEPSFSEIEELFLQTTFDKMDFLTSVKDCAVEMHHPHMGFSVHQNLKDSLSRYLLELSEQLLSNYSSEITNSEILVRCSSLLVGVLGCYSYLGVIAEEEACKSELFHKAKSLMQCAGESITLFKNKTNEDSRIGSLRSMMHLCTSYLCNCIKQ

In [6]:
candidate_df = pd.DataFrame(rows)

candidate_df.head()

,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence
0,XP_040832827.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...
1,XP_040832828.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...
2,XP_040832829.1,ATM,Ochotona curzoniae,121154090,X2,2935,0,0,METVKDSSNSSTYGADCSNILLKDILSVRKYWCEISQQQWLELFSV...
3,XP_040832830.1,ATM,Ochotona curzoniae,121154090,X3,2623,0,0,MILYQLLSQRRRGERTPYVLRCLTEVALCQGKKSNLETSQQSDLLK...
4,XP_059322021.1,ATM,Ammospiza nelsoni,132069672,<NA>,3066,0,0,MSLALHDLLVCCRRLENEKAVERRKEIENFKRLLRDPETVLQLDRN...


In [7]:
def get_accession_priority(accession):
    if accession.startswith("NP_"):
        return 0
    if accession.startswith("XP_"):
        return 1
    else:
        return 2

# check if human protein exists
human_reference = candidate_df[candidate_df["accession"] == "NP_000042.3"]

if human_reference.empty:
    raise ValueError("Human reference NP_000042.3 was not found.")

human_length = human_reference.iloc[0]["sequence_length"]
print(human_length)

# additional metadata
candidate_df["accession_priority"] = candidate_df["accession"].apply(get_accession_priority)
candidate_df["length_difference"] = (candidate_df["sequence_length"] - human_length).abs()
candidate_df["length_ratio"] = candidate_df["sequence_length"] / human_length

# assign length status 
candidate_df["length_status"] ="Fail"

candidate_df.loc[
    candidate_df["length_ratio"].between(0.75, 1.25),
    "length_status"
] = "Review"

candidate_df.loc[
    candidate_df["length_ratio"].between(0.90, 1.10),
    "length_status"
] = "Pass"

# assign basic qc passes
candidate_df["passes_basic_qc"] = (
    candidate_df["organism"].notna()
    & (candidate_df["gene_symbol"].str.upper() == "ATM")
    & (candidate_df["length_status"] != "Fail")
    & (candidate_df["ambiguous_x_count"] <= 5)
    & (candidate_df["stop_count"] == 0)
)

candidate_df.head()


3056


,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence,accession_priority,length_difference,length_ratio,length_status,passes_basic_qc
0,XP_040832827.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...,1,3,1.000982,Pass,True
1,XP_040832828.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...,1,3,1.000982,Pass,True
2,XP_040832829.1,ATM,Ochotona curzoniae,121154090,X2,2935,0,0,METVKDSSNSSTYGADCSNILLKDILSVRKYWCEISQQQWLELFSV...,1,121,0.960406,Pass,True
3,XP_040832830.1,ATM,Ochotona curzoniae,121154090,X3,2623,0,0,MILYQLLSQRRRGERTPYVLRCLTEVALCQGKKSNLETSQQSDLLK...,1,433,0.858312,Review,True
4,XP_059322021.1,ATM,Ammospiza nelsoni,132069672,<NA>,3066,0,0,MSLALHDLLVCCRRLENEKAVERRKEIENFKRLLRDPETVLQLDRN...,1,10,1.003272,Pass,True


In [8]:
selected_df = (candidate_df.loc[candidate_df["passes_basic_qc"]]
    .sort_values(
        by =[
            "organism",
            "accession_priority",
            "ambiguous_x_count",
            "length_difference",
        ]
    )
    .drop_duplicates(
        subset=["organism"],
        keep="first"
    )
    .reset_index(drop=True)
    .copy()
)

print(f"selected {len(selected_df)}")
print(f"candidates {len(candidate_df)}")
selected_df.head()

selected 827
candidates 2603


,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence,accession_priority,length_difference,length_ratio,length_status,passes_basic_qc
0,XP_083981534.1,atm,Abramis brama,311090226,<NA>,3102,0,0,MSLALHELLLCCRGLENEKATERKKEVDRFRRLIRSPDTVEELDRT...,1,46,1.015052,Pass,True
1,XP_022052894.2,atm,Acanthochromis polyacanthus,110953284,X1,3092,0,0,MSLALNDLLLCCRGLESDKATERKKETERFRLLLQSPETIQELDRT...,1,36,1.011780,Pass,True
2,XP_036940904.1,atm,Acanthopagrus latus,119011683,<NA>,3090,0,0,MSLALHDLMVCCRGLESDKATERKKVAEQFRRLIRTPETVQELDRS...,1,34,1.011126,Pass,True
3,XP_026892237.2,ATM,Acinonyx jubatus,106983963,X3,3057,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIRDPETVQHLDRH...,1,1,1.000327,Pass,True
4,XP_084640426.1,ATM,Acomys minous,311598838,1,3054,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIQDPETVQNLDRH...,1,2,0.999346,Pass,True


In [9]:
# protein fasta into a csv
candidate_output = (
    project_root
    / "data"
    / "processed"
    / "atm_ortholog_candidates.csv"
)

# protein filtered fasta into a csv
selected_output = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.csv"
)

candidate_df.to_csv(candidate_output, index=False)
selected_df.to_csv(selected_output, index=False)

print(f"Candidates saved to: {candidate_output}")
print(f"Selected saved to: {selected_output}")

Candidates saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_ortholog_candidates.csv
Selected saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_representative_proteins.csv


In [10]:
# recheck if human_reference survived filtering
human_reference = selected_df[
    selected_df["accession"] == "NP_000042.3"
]

print("Human reference records found:", len(human_reference))

if human_reference.empty:
    raise ValueError(
        "Human ATM reference NP_000042.3 was not selected."
    )

Human reference records found: 1


In [11]:
selected_records = []

for row in selected_df.itertuples(index=False):
    fasta_record = SeqRecord(
        seq = Seq(row.sequence),
        id = row.accession,
        description = f"ATM [organism={row.organism}] [GeneID={row.gene_id}] [isoform={row.isoform}]"
    )
    selected_records.append(fasta_record)

for _ in selected_records[:3]:
    print(_)

ID: XP_083981534.1
Name: <unknown name>
Description: ATM [organism=Abramis brama] [GeneID=311090226] [isoform=<NA>]
Number of features: 0
Seq('MSLALHELLLCCRGLENEKATERKKEVDRFRRLIRSPDTVEELDRTSGCKASKG...AWV')
ID: XP_022052894.2
Name: <unknown name>
Description: ATM [organism=Acanthochromis polyacanthus] [GeneID=110953284] [isoform=X1]
Number of features: 0
Seq('MSLALNDLLLCCRGLESDKATERKKETERFRLLLQSPETIQELDRTSGPKSKGS...AWV')
ID: XP_036940904.1
Name: <unknown name>
Description: ATM [organism=Acanthopagrus latus] [GeneID=119011683] [isoform=<NA>]
Number of features: 0
Seq('MSLALHDLMVCCRGLESDKATERKKVAEQFRRLIRTPETVQELDRSSGLKDKGS...AWV')


In [12]:
fasta_output = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.fasta"
)

written_count = SeqIO.write(
    selected_records,
    fasta_output,
    "fasta"
)

print(f"Wrote {written_count} protein sequences.")
print(f"Saved to: {fasta_output}")

Wrote 827 protein sequences.
Saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_representative_proteins.fasta


In [13]:
saved_records = list(
    SeqIO.parse(fasta_output, "fasta")
)

print("Expected records:", len(selected_df))
print("Saved FASTA records:", len(saved_records))

Expected records: 827
Saved FASTA records: 827
